# GPT CE Analysis — interactive plotting
Remake of the CE loss figure with DiT-matched colour convention.
- **Train**: blue `#2166ac`
- **Valid (novel)**: red `#d73027`
- **Boolean cube**: gray dashed `#555555`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

%matplotlib inline
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42

In [ ]:
SAVEROOT = (
    "/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang/"
    "DL_Projects/DiffusionParityLearning"
)
FIGDIR = (
    "/n/home12/binxuwang/Github/DiffusionAttnConsistency/figures/"
    "GPT_parity_learn_dissection"
)
os.makedirs(FIGDIR, exist_ok=True)

EXP_NAMES = [
    "GPT_mini_parity_N4096_D36_G6_even_lr1e4",
    "GPT_mini_parity_N4096_D36_G6_even_wd1e2",
]
GROUP_SIZE = 6
N_EVAL     = 4096

In [ ]:
# ── Import the plotting helpers from the script ──────────────────────────────
from scripts.plot_GPT_CE_analysis import (
    plot_ce_figure, load_ce_data,
    SPLIT_STYLES, HMAP_CMAPS, HMAP_TITLES,
    make_step_axis, heatmap_step_ticks, group_boundary_lines,
)
from scripts.plot_tb_curves import load_tb_scalars

## Quick one-shot: generate both runs

In [ ]:
for exp_name in EXP_NAMES:
    exp_dir  = os.path.join(SAVEROOT, exp_name)
    short    = exp_name.replace("GPT_mini_parity_N4096_D36_", "").replace("_even", "")
    out_base = os.path.join(FIGDIR, f"GPT_G6_CE_analysis_{short}")

    fig = plot_ce_figure(exp_dir, group_size=GROUP_SIZE, n_eval=N_EVAL)
    fig.savefig(out_base + ".pdf", bbox_inches="tight")
    fig.savefig(out_base + ".png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"Saved → {out_base}.{{pdf,png}}")

## Raw training stats (Accuracy, Mem Ratio) — log step scale

In [ ]:
# Tags available in GPT tensorboard:
#   Eval/Sample_Accuracy   Eval/PerGroup_Accuracy
#   Eval/Sample_Mem_Ratio  Eval/BitGroup_Mem_Ratio
#   Training/Loss_Step     Training/Loss_Avg

STAT_TAGS = [
    "Eval/Sample_Accuracy",
    "Eval/PerGroup_Accuracy",
    "Eval/Sample_Mem_Ratio",
    "Eval/BitGroup_Mem_Ratio",
    "Training/Loss_Step",
]
STAT_LABELS = [
    "Sample Accuracy",
    "Per-Group Accuracy",
    "Sample Mem Ratio",
    "BitGroup Mem Ratio",
    "Train Loss (step)",
]
YHLINES = {
    "Eval/Sample_Accuracy":   0.9,
    "Eval/PerGroup_Accuracy": 0.9,
    "Eval/Sample_Mem_Ratio":  0.5,
    "Eval/BitGroup_Mem_Ratio":0.5,
}

In [ ]:
exp_name = EXP_NAMES[0]   # ← change me
exp_dir  = os.path.join(SAVEROOT, exp_name)
tb_dir   = os.path.join(exp_dir, "tensorboard")

tb_data = load_tb_scalars(tb_dir, STAT_TAGS)

ncols = 3
nrows = (len(STAT_TAGS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), dpi=130)
axes = axes.flatten()

for i, (tag, lbl) in enumerate(zip(STAT_TAGS, STAT_LABELS)):
    ax = axes[i]
    if tag in tb_data:
        steps = np.array(tb_data[tag]["steps"])
        vals  = np.array(tb_data[tag]["vals"])
        ax.plot(steps + 1, vals, lw=1.5, color="steelblue")
    if tag in YHLINES:
        ax.axhline(YHLINES[tag], color="gray", lw=0.9, ls="--")
    ax.set_xscale("log")
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("Step", fontsize=10)

# hide unused panels
for j in range(len(STAT_TAGS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(exp_name, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Overlay two runs on the same axes for comparison ─────────────────────────
run_colors = ["#2166ac", "#d73027"]   # ← one colour per run

all_tb = {
    exp: load_tb_scalars(os.path.join(SAVEROOT, exp, "tensorboard"), STAT_TAGS)
    for exp in EXP_NAMES
}

ncols = 3
nrows = (len(STAT_TAGS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), dpi=130)
axes = axes.flatten()

for i, (tag, lbl) in enumerate(zip(STAT_TAGS, STAT_LABELS)):
    ax = axes[i]
    for exp, col in zip(EXP_NAMES, run_colors):
        d = all_tb[exp]
        if tag in d:
            steps = np.array(d[tag]["steps"])
            vals  = np.array(d[tag]["vals"])
            short = exp.replace("GPT_mini_parity_N4096_D36_", "").replace("_even", "")
            ax.plot(steps + 1, vals, lw=1.5, color=col, label=short)
    if tag in YHLINES:
        ax.axhline(YHLINES[tag], color="gray", lw=0.9, ls="--")
    ax.set_xscale("log")
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("Step", fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)

for j in range(len(STAT_TAGS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("GPT baseline runs — raw stats", fontsize=12)
plt.tight_layout()
plt.show()

## Interactive / custom CE tweaking
Copy-paste the cell below and modify as needed.

In [ ]:
import json

exp_name = EXP_NAMES[0]   # ← change me
exp_dir  = os.path.join(SAVEROOT, exp_name)

d       = load_ce_data(exp_dir, n_eval=N_EVAL)
epochs  = d['epochs']
splits  = ['train', 'valid_novel', 'boolean_cube']
loss    = {s: d[f'loss_{s}']     for s in splits}
pos_loss= {s: d[f'pos_loss_{s}'] for s in splits}   # (C, 36)
n_pos   = pos_loss['train'].shape[1]

args_dict = json.loads(str(d['args_json']))
exp_tag   = args_dict.get('exp_name', os.path.basename(exp_dir))
lr        = args_dict.get('lr',            '?')
wd        = args_dict.get('weight_decay',  '?')

# ── figure layout ────────────────────────────────────────────────────────────
figsize = (13, 7)    # ← tweak
fig = plt.figure(figsize=figsize, dpi=150)
gs  = fig.add_gridspec(2, 3, height_ratios=[1.2, 1.4], hspace=0.45, wspace=0.30)

# ── top: CE loss curves ───────────────────────────────────────────────────────
ax_loss = fig.add_subplot(gs[0, :])
for s in splits:
    sty = SPLIT_STYLES[s]
    ax_loss.plot(epochs, loss[s], **sty)

ax_loss.axhline(np.log(2), color='gray', lw=0.9, ls=':', label=f'log(2)={np.log(2):.3f}')
ax_loss.set_ylabel('CE loss', fontsize=11)
ax_loss.set_title(f'{exp_tag}  (lr={lr}, wd={wd})', fontsize=11)
ax_loss.legend(loc='upper left', fontsize=10)
make_step_axis(ax_loss, epochs)

# ── bottom: per-position heatmaps ────────────────────────────────────────────
vmax = max(np.nanpercentile(pos_loss[s], 97) for s in splits)

hax = [fig.add_subplot(gs[1, c]) for c in range(3)]
for col, s in enumerate(splits):
    ax  = hax[col]
    mat = pos_loss[s].T   # (36, C)
    im  = ax.imshow(mat, aspect='auto', origin='lower',
                    cmap=HMAP_CMAPS[s],
                    norm=Normalize(vmin=0, vmax=vmax),
                    interpolation='nearest')
    group_boundary_lines(ax, GROUP_SIZE, n_pos, orientation='horizontal')
    heatmap_step_ticks(ax, epochs)
    ax.set_ylabel('Bit position' if col == 0 else '', fontsize=10)
    ax.set_title(HMAP_TITLES[s], fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.85, label='CE')

fig.suptitle('GPT CE analysis', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()